# Log e Registro do modelo via MLFlow Model Serving

## Log do modelo como parte do experimento `heart-disease-prediction`

In [0]:
import warnings, mlflow, joblib
import pandas as pd
from mlflow.models.signature import infer_signature

warnings.filterwarnings('ignore')

# Carregar o modelo via joblib
model = joblib.load('models/best_random_forest.joblib')

# Criar experimento
mlflow.set_experiment('/heart-disease-prediction')

# Carregar dataset de exemplo
path = "../../data/heart_disease_uci_preprocessed.csv"
df = pd.read_csv(path)

# Selecionar uma amostra de exemplo
X_train = df.drop(columns=['target'])[:5]
y_train = df['target'][:5]

# Adicionar assinatura do modelo
model_signature = infer_signature(X_train, y_train)

# Log do modelo no MLFlow
mlflow_model = mlflow.sklearn.log_model(model, 'model', signature=model_signature)
mlflow.end_run()

## Registro do modelo

In [0]:
from mlflow import MlflowClient

# Set Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Registrar modelo
model_registered = mlflow.register_model(model_uri = mlflow_model.model_uri,
                                         name = 'heart-disease-model')


# Criar alias Staging/Production e setar ao modelo
MlflowClient().set_registered_model_alias(model_registered.name, 
                                  "Production", 
                                  model_registered.version)


## Configurar o modelo atual como Staging

In [0]:
from mlflow import MlflowClient
client = MlflowClient()

# Criar alias Staging e setar ao modelo
client.set_registered_model_alias(model_registered.name, 
                                  "Staging", 
                                  model_registered.version)